# NMF model

In [1]:
!pip install scikit-surprise

In [1]:
from surprise import NMF
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy

In [5]:
# Read the course rating dataset with columns user item rating
reader = Reader(line_format='item user rating', sep=',', skip_lines=1, rating_scale=(2, 3))

# Load the dataset from the CSV file
course_dataset = Dataset.load_from_file("/home/uzokmurod/Desktop/amaliyot/book recommendation sysrem/data/raw/ratings.csv", reader=reader)

In [6]:
trainset, testset = train_test_split(course_dataset, test_size=.3)

In [7]:
print(f"Total {trainset.n_users} users and {trainset.n_items} items in the trainingset")

Total 52542 users and 10000 items in the trainingset


In [8]:
model = NMF(verbose=True, random_state=42, init_low=0.5, init_high=5)
model.fit(trainset)
prediction = model.test(testset)
accuracy.rmse(prediction)

Processing epoch 0
Processing epoch 1
Processing epoch 2
Processing epoch 3
Processing epoch 4
Processing epoch 5
Processing epoch 6
Processing epoch 7
Processing epoch 8
Processing epoch 9
Processing epoch 10
Processing epoch 11
Processing epoch 12
Processing epoch 13
Processing epoch 14
Processing epoch 15
Processing epoch 16
Processing epoch 17
Processing epoch 18
Processing epoch 19
Processing epoch 20
Processing epoch 21
Processing epoch 22
Processing epoch 23
Processing epoch 24
Processing epoch 25
Processing epoch 26
Processing epoch 27
Processing epoch 28
Processing epoch 29
Processing epoch 30
Processing epoch 31
Processing epoch 32
Processing epoch 33
Processing epoch 34
Processing epoch 35
Processing epoch 36
Processing epoch 37
Processing epoch 38
Processing epoch 39
Processing epoch 40
Processing epoch 41
Processing epoch 42
Processing epoch 43
Processing epoch 44
Processing epoch 45
Processing epoch 46
Processing epoch 47
Processing epoch 48
Processing epoch 49
RMSE: 1.30

1.3042458913185209

In [9]:
def get_top_n_recommendations(model, trainset, user_id, n=5):
    # Get list of all item raw ids
    all_items = trainset._raw2inner_id_items.keys()

    # Get internal user id (surprise uses inner ids)
    try:
        inner_uid = trainset.to_inner_uid(user_id)
        seen_items = [trainset.to_raw_iid(iid) for (iid, _) in trainset.ur[inner_uid]]
    except ValueError:
        # New user — hasn't rated anything yet
        seen_items = []

    # Filter out items already rated by user
    unseen_items = [item for item in all_items if item not in seen_items]

    # Predict ratings for all unseen items
    predictions = [model.predict(user_id, item_id) for item_id in unseen_items]

    # Sort by estimated rating
    top_predictions = sorted(predictions, key=lambda x: x.est, reverse=True)[:n]

    # Return (item_id, predicted_rating)
    return [(pred.iid, round(pred.est, 2)) for pred in top_predictions]


In [10]:
user_id = '42'  # use the correct ID from your dataset
top_n = get_top_n_recommendations(model, trainset, user_id, n=5)

print(f"Top recommendations for user {user_id}:")
for item_id, rating in top_n:
    print(f"Item: {item_id}, Predicted Rating: {rating}")


Top recommendations for user 42:
Item: 7937, Predicted Rating: 3
Item: 5322, Predicted Rating: 3
Item: 4538, Predicted Rating: 3
Item: 1251, Predicted Rating: 3
Item: 294, Predicted Rating: 3
